# 02 — Exploração e Tratamento dos Dados do MapBiomas Solo

## Projeto AgroESG — Correção de Escopo

Este notebook prepara os dados de carbono orgânico do solo do MapBiomas Solo para o recorte territorial do projeto: **Centro-Oeste e Sul**.

O MapBiomas Solo não possui variável de cultura; portanto, nesta fonte o filtro aplicado é territorial. A associação com a **soja** ocorre posteriormente pela integração com a PAM/IBGE.

Pipeline: `RAW → PROCESSED → CURATED`  
Período: 2019–2024.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
BASE_DIR = Path.cwd().parent

RAW_DIR = (
    BASE_DIR
    / "data"
    / "raw"
    / "mapbiomas_solo"
)

PROCESSED_DIR = (
    BASE_DIR
    / "data"
    / "databases_processed"
    / "mapbiomas_solo"
)

CURATED_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "mapbiomas_solo"
)

print("BASE_DIR:", BASE_DIR)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("CURATED_DIR:", CURATED_DIR)


In [ ]:
PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CURATED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Pastas verificadas/criadas.")


In [ ]:
arquivos_csv = list(
    RAW_DIR.glob("*.csv")
)

print("Arquivos CSV encontrados:")

for arquivo in arquivos_csv:
    print("-", arquivo.name)


In [ ]:
assert len(arquivos_csv) == 1, (
    f"Esperado 1 CSV em RAW, mas foram encontrados {len(arquivos_csv)}."
)

arquivo_solo = arquivos_csv[0]

print("Arquivo selecionado:")
print(arquivo_solo)


In [ ]:
with open(
    arquivo_solo,
    "r",
    encoding="utf-8-sig"
) as arquivo:

    for _ in range(5):
        print(arquivo.readline().strip())


In [ ]:
solo_raw = pd.read_csv(
    arquivo_solo,
    encoding="utf-8-sig",
    dtype={
        "CD_MUN": "string"
    }
)

solo_raw.head()


In [ ]:
print(
    "Dimensão da base:",
    solo_raw.shape
)

print(
    "Linhas:",
    solo_raw.shape[0]
)

print(
    "Colunas:",
    solo_raw.shape[1]
)


In [ ]:
print("Colunas:")

for coluna in solo_raw.columns:
    print("-", coluna)


In [ ]:
solo_raw.dtypes


In [ ]:
print(
    "Municípios:",
    solo_raw["CD_MUN"].nunique()
)

print(
    "UFs:",
    solo_raw["SIGLA_UF"].nunique()
)

print(
    "Regiões:",
    sorted(
        solo_raw["SIGLA_RG"].unique()
    )
)


In [ ]:
duplicados_municipios = (
    solo_raw
    .duplicated(
        subset=["CD_MUN"]
    )
    .sum()
)

print(
    "Municípios duplicados:",
    duplicados_municipios
)


In [ ]:
nulos_raw = (
    solo_raw
    .isna()
    .sum()
    .sort_values(
        ascending=False
    )
)

nulos_raw


In [ ]:
colunas_carbono = [
    coluna
    for coluna in solo_raw.columns
    if coluna.startswith("carbon_")
]

solo_raw[
    solo_raw[colunas_carbono]
    .isna()
    .any(axis=1)
]


In [ ]:
# 15. Estatísticas descritivas das colunas de carbono

solo_raw[
    colunas_carbono
].describe().T


In [ ]:
# 16. Verificar valores negativos

for coluna in colunas_carbono:
    negativos = (solo_raw[coluna] < 0).sum()

    print(
        coluna,
        "-> negativos:",
        negativos
    )


## Diagnóstico inicial da camada RAW

A camada RAW é inspecionada apenas para compreender estrutura, tipos, valores ausentes, duplicidades e distribuição do carbono. O arquivo original permanece preservado sem alterações.

O recorte territorial do projeto é aplicado na construção da camada `Processed`, mantendo somente **Centro-Oeste** e **Sul**. Valores ausentes de carbono continuam preservados como `NaN`, pois ausência de informação não equivale a estoque zero.


In [ ]:
solo_processado = solo_raw.copy()

solo_processado.head()


## 17. Construção da camada PROCESSED

A partir desta etapa, é criada uma cópia da base RAW para aplicação das transformações de limpeza, normalização e padronização.

A camada RAW permanece preservada sem alterações.


In [ ]:
solo_processado = solo_processado.rename(
    columns={
        "CD_MUN": "codigo_ibge",
        "NM_MUN": "municipio",
        "SIGLA_UF": "uf",
        "SIGLA_RG": "regiao",
        "AREA_KM2": "area_km2"
    }
)

solo_processado.columns


In [ ]:
solo_processado["codigo_ibge"] = (
    solo_processado["codigo_ibge"]
    .astype("string")
    .str.strip()
)

solo_processado["municipio"] = (
    solo_processado["municipio"]
    .astype("string")
    .str.strip()
)

solo_processado["uf"] = (
    solo_processado["uf"]
    .astype("string")
    .str.strip()
)

solo_processado["regiao"] = (
    solo_processado["regiao"]
    .astype("string")
    .str.strip()
)

solo_processado["area_km2"] = pd.to_numeric(
    solo_processado["area_km2"],
    errors="coerce"
)

solo_processado.dtypes


In [ ]:
colunas_id = [
    "codigo_ibge",
    "municipio",
    "uf",
    "regiao",
    "area_km2"
]

colunas_carbono = [
    "carbon_2019",
    "carbon_2020",
    "carbon_2021",
    "carbon_2022",
    "carbon_2023",
    "carbon_2024"
]

solo_processado = solo_processado.melt(
    id_vars=colunas_id,
    value_vars=colunas_carbono,
    var_name="carbono_ano",
    value_name="carbono_solo_t_ha"
)

solo_processado.head(10)


In [ ]:
solo_processado["ano"] = (
    solo_processado["carbono_ano"]
    .str.extract(r"(\d{4})")[0]
    .astype("int64")
)

solo_processado = solo_processado.drop(
    columns="carbono_ano"
)

solo_processado.head()


In [ ]:
solo_processado = solo_processado[
    [
        "codigo_ibge", "municipio", "uf", "regiao", "ano",
        "area_km2", "carbono_solo_t_ha"
    ]
].copy()

# Recorte territorial do projeto ainda usando as siglas originais da região
solo_processado = solo_processado[
    solo_processado["regiao"].isin(["CO", "S"])
].copy()

print("Registros após recorte Centro-Oeste/Sul:", len(solo_processado))


In [ ]:
print(
    "Dimensão processada:",
    solo_processado.shape
)

print(
    "Municípios:",
    solo_processado["codigo_ibge"].nunique()
)

print(
    "Anos:",
    sorted(
        solo_processado["ano"].unique()
    )
)

print(
    "UFs:",
    solo_processado["uf"].nunique()
)

print(
    "Regiões:",
    solo_processado["regiao"].unique()
)


In [ ]:
duplicados = (
    solo_processado
    .duplicated(
        subset=[
            "codigo_ibge",
            "ano"
        ]
    )
    .sum()
)

print(
    "Duplicados na chave município-ano:",
    duplicados
)


In [ ]:
solo_processado.isna().sum().sort_values(
    ascending=False
)


In [ ]:
solo_processado.isna().sum().sort_values(
    ascending=False
)


In [ ]:
negativos = (
    solo_processado["carbono_solo_t_ha"] < 0
).sum()

print(
    "Valores negativos de carbono:",
    negativos
)


In [ ]:
solo_processado[
    "carbono_solo_t_ha"
].describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99
    ]
)


In [ ]:
solo_processado["status_dado"] = (
    solo_processado[
        "carbono_solo_t_ha"
    ]
    .notna()
    .map({
        True: "disponivel",
        False: "dados_nao_captados"
    })
)

solo_processado[
    "status_dado"
].value_counts()


In [ ]:
assert solo_processado.duplicated(subset=["codigo_ibge", "ano"]).sum() == 0
assert solo_processado["ano"].between(2019, 2024).all()
assert solo_processado["carbono_solo_t_ha"].dropna().ge(0).all()
assert set(solo_processado["regiao"].dropna().unique()) == {"CO", "S"}
assert set(solo_processado["uf"].dropna().unique()) <= {"DF", "GO", "MT", "MS", "PR", "RS", "SC"}

print("✅ Validação da camada PROCESSED concluída.")


## 31. Exportação da camada PROCESSED

A camada `Processed` é exportada já restrita aos municípios do **Centro-Oeste** e **Sul**, mantendo o período 2019–2024.


In [ ]:
BASE_DIR = Path.cwd().parent

PROCESSED_DIR = (BASE_DIR / "data" / "databases_processed" / "mapbiomas_solo")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

arquivo_processed = (
    PROCESSED_DIR
    / "mapbiomas_solo_municipios_centro_oeste_sul_2019_2024_processed.csv"
)

solo_processado.to_csv(arquivo_processed, index=False, encoding="utf-8-sig")
print("Arquivo salvo em:", arquivo_processed)


## 32. Construção da camada CURATED

A camada `Curated` preserva apenas os municípios do **Centro-Oeste** e **Sul**, converte as siglas regionais para os nomes completos e mantém explicitamente registros sem informação de carbono.


In [ ]:
solo_curated = (
    solo_processado[
        solo_processado["ano"].between(
            2019,
            2024
        )
    ]
    .copy()
)

solo_curated.head()


In [ ]:
mapa_regioes = {
    "S": "Sul",
    "CO": "Centro-Oeste"
}

solo_curated["regiao"] = solo_curated["regiao"].replace(mapa_regioes).astype("string")
solo_curated = solo_curated[solo_curated["regiao"].isin(["Centro-Oeste", "Sul"])].copy()

solo_curated["regiao"].value_counts()


In [ ]:
print(
    "Dimensão:",
    solo_curated.shape
)

print(
    "Municípios:",
    solo_curated["codigo_ibge"].nunique()
)

print(
    "Anos:",
    sorted(
        solo_curated["ano"].unique()
    )
)

print(
    "UFs:",
    solo_curated["uf"].nunique()
)

print(
    "Regiões:",
    solo_curated["regiao"].unique()
)

print(
    "\nStatus dos dados:"
)

print(
    solo_curated["status_dado"]
    .value_counts(dropna=False)
)


In [ ]:
assert solo_curated.duplicated(subset=["codigo_ibge", "ano"]).sum() == 0
assert solo_curated["ano"].between(2019, 2024).all()
assert solo_curated["uf"].nunique() == 7
assert set(solo_curated["regiao"].unique()) == {"Centro-Oeste", "Sul"}
assert set(solo_curated["uf"].unique()) <= {"DF", "GO", "MT", "MS", "PR", "RS", "SC"}
assert solo_curated["carbono_solo_t_ha"].dropna().ge(0).all()

print("✅ Validação da camada CURATED concluída.")


In [ ]:
CURATED_DIR = (
    BASE_DIR
    / "data"
    / "databases_curated"
    / "mapbiomas_solo"
)
CURATED_DIR.mkdir(parents=True, exist_ok=True)

arquivo_curated = (
    CURATED_DIR
    / "mapbiomas_solo_municipios_centro_oeste_sul_2019_2024_curated.csv"
)

solo_curated.to_csv(arquivo_curated, index=False, encoding="utf-8-sig")

print("PROCESSSED:")
print(arquivo_processed)
print("\nCURATED:")
print(arquivo_curated)
print("\nExistem?", arquivo_processed.exists(), arquivo_curated.exists())


## 37. Evolução temporal do carbono orgânico do solo

Nesta etapa é analisado o comportamento médio do estoque de carbono orgânico do solo entre 2019 e 2024.

Os cálculos são realizados exclusivamente sobre registros com informação disponível, preservando separadamente os casos classificados como `dados_nao_captados`.


In [ ]:
carbono_anual = (
    solo_curated
    .groupby(
        "ano",
        as_index=False
    )
    ["carbono_solo_t_ha"]
    .mean()
)

carbono_anual


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    carbono_anual["ano"],
    carbono_anual["carbono_solo_t_ha"],
    marker="o"
)

plt.title(
    "Evolução do carbono orgânico médio do solo - Brasil"
)

plt.xlabel("Ano")
plt.ylabel("Carbono orgânico do solo (t/ha)")

plt.grid(alpha=0.3)

plt.show()


## 38. Carbono orgânico médio do solo por região

A comparação regional é realizada exclusivamente entre **Centro-Oeste** e **Sul**, conforme o escopo corrigido.


In [ ]:
carbono_regiao = (
    solo_curated
    .groupby(
        "regiao",
        as_index=False
    )
    ["carbono_solo_t_ha"]
    .mean()
    .sort_values(
        "carbono_solo_t_ha",
        ascending=False
    )
)

carbono_regiao


In [ ]:
plt.figure(figsize=(9, 5))

plt.bar(
    carbono_regiao["regiao"],
    carbono_regiao["carbono_solo_t_ha"]
)

plt.title(
    "Carbono orgânico médio do solo por região"
)

plt.xlabel("Região")
plt.ylabel("Carbono orgânico do solo (t/ha)")

plt.xticks(rotation=20)

plt.show()


In [ ]:
carbono_regiao_ano = (
    solo_curated
    .groupby(
        [
            "ano",
            "regiao"
        ],
        as_index=False
    )
    ["carbono_solo_t_ha"]
    .mean()
)

carbono_regiao_ano.head(15)


In [ ]:
plt.figure(figsize=(10, 6))

for regiao in carbono_regiao_ano["regiao"].unique():

    dados = carbono_regiao_ano[
        carbono_regiao_ano["regiao"] == regiao
    ]

    plt.plot(
        dados["ano"],
        dados["carbono_solo_t_ha"],
        marker="o",
        label=regiao
    )

plt.title(
    "Evolução do carbono orgânico do solo por região"
)

plt.xlabel("Ano")
plt.ylabel("Carbono orgânico do solo (t/ha)")

plt.legend(
    title="Região"
)

plt.grid(alpha=0.3)

plt.show()


In [ ]:
carbono_uf = (
    solo_curated
    .groupby(
        [
            "uf",
            "regiao"
        ],
        as_index=False
    )
    ["carbono_solo_t_ha"]
    .mean()
    .sort_values(
        "carbono_solo_t_ha",
        ascending=False
    )
)

carbono_uf


In [ ]:
carbono_variacao = (
    solo_curated
    .pivot_table(
        index=[
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao"
        ],
        columns="ano",
        values="carbono_solo_t_ha"
    )
    .reset_index()
)

carbono_variacao.columns.name = None

carbono_variacao.head()


In [ ]:
carbono_variacao[
    "variacao_carbono_t_ha_2019_2024"
] = (
    carbono_variacao[2024]
    - carbono_variacao[2019]
)


In [ ]:
carbono_variacao[
    "variacao_carbono_pct_2019_2024"
] = (
    (
        carbono_variacao[2024]
        - carbono_variacao[2019]
    )
    / carbono_variacao[2019]
    * 100
)


In [ ]:
print(
    "Municípios com carbono = 0 em 2019:",
    (carbono_variacao[2019] == 0).sum()
)

print(
    "Municípios sem informação em 2019:",
    carbono_variacao[2019].isna().sum()
)

print(
    "Municípios sem informação em 2024:",
    carbono_variacao[2024].isna().sum()
)


In [ ]:
import numpy as np

carbono_variacao[
    "variacao_carbono_pct_2019_2024"
] = np.where(
    carbono_variacao[2019] > 0,
    (
        (
            carbono_variacao[2024]
            - carbono_variacao[2019]
        )
        / carbono_variacao[2019]
        * 100
    ),
    np.nan
)


In [ ]:
print(
    "Percentuais infinitos:",
    np.isinf(
        carbono_variacao[
            "variacao_carbono_pct_2019_2024"
        ]
    ).sum()
)

print(
    "Percentuais ausentes:",
    carbono_variacao[
        "variacao_carbono_pct_2019_2024"
    ].isna().sum()
)


## 42. Tratamento de valor inicial igual a zero

Foi identificado um município com estoque de carbono igual a zero em 2019.

Como a variação percentual exige divisão pelo valor inicial, esse registro não possui variação percentual matematicamente válida. Por isso, o campo percentual é mantido como ausente (`NaN`), enquanto a variação absoluta em t/ha permanece disponível.


In [ ]:
carbono_variacao[
    carbono_variacao[2019] == 0
][
    [
        "codigo_ibge",
        "municipio",
        "uf",
        "regiao",
        2019,
        2024,
        "variacao_carbono_t_ha_2019_2024",
        "variacao_carbono_pct_2019_2024"
    ]
]


In [ ]:
carbono_variacao[
    [
        "variacao_carbono_t_ha_2019_2024",
        "variacao_carbono_pct_2019_2024"
    ]
].describe(
    percentiles=[
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        0.99
    ]
)


In [ ]:
def classificar_variacao(valor):

    if pd.isna(valor):
        return "nao_calculavel"

    elif valor > 1:
        return "aumento"

    elif valor < -1:
        return "reducao"

    else:
        return "estavel"


carbono_variacao["classe_variacao_2019_2024"] = (
    carbono_variacao[
        "variacao_carbono_pct_2019_2024"
    ]
    .apply(classificar_variacao)
)


In [ ]:
carbono_variacao[
    "classe_variacao_2019_2024"
].value_counts()


In [ ]:
distribuicao_variacao = (
    carbono_variacao[
        "classe_variacao_2019_2024"
    ]
    .value_counts()
    .rename_axis("classe")
    .reset_index(name="municipios")
)

distribuicao_variacao[
    "percentual"
] = (
    distribuicao_variacao["municipios"]
    / distribuicao_variacao["municipios"].sum()
    * 100
)

distribuicao_variacao


In [ ]:
maiores_reducoes = (
    carbono_variacao
    .dropna(
        subset=[
            "variacao_carbono_pct_2019_2024"
        ]
    )
    .sort_values(
        "variacao_carbono_pct_2019_2024",
        ascending=True
    )
    [
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao",
            2019,
            2024,
            "variacao_carbono_t_ha_2019_2024",
            "variacao_carbono_pct_2019_2024"
        ]
    ]
    .head(20)
)

maiores_reducoes


In [ ]:
maiores_aumentos = (
    carbono_variacao
    .dropna(
        subset=[
            "variacao_carbono_pct_2019_2024"
        ]
    )
    .sort_values(
        "variacao_carbono_pct_2019_2024",
        ascending=False
    )
    [
        [
            "codigo_ibge",
            "municipio",
            "uf",
            "regiao",
            2019,
            2024,
            "variacao_carbono_t_ha_2019_2024",
            "variacao_carbono_pct_2019_2024"
        ]
    ]
    .head(20)
)

maiores_aumentos


In [ ]:
variacao_regiao = (
    carbono_variacao
    .groupby(
        "regiao",
        as_index=False
    )
    .agg(
        carbono_2019_medio=(2019, "mean"),
        carbono_2024_medio=(2024, "mean"),
        variacao_media_t_ha=(
            "variacao_carbono_t_ha_2019_2024",
            "mean"
        ),
        variacao_media_pct=(
            "variacao_carbono_pct_2019_2024",
            "mean"
        )
    )
)

variacao_regiao.sort_values(
    "variacao_media_pct"
)


# Conclusão

O MapBiomas Solo foi estruturado para o período 2019–2024 e restringido aos municípios das regiões **Centro-Oeste** e **Sul**.

Como essa fonte não possui cultura agrícola, a seleção de **soja** é feita no Notebook 03 por meio da integração com a PAM/IBGE. As camadas `Processed` e `Curated` ficam prontas para esse cruzamento municipal por `codigo_ibge` e `ano`.


In [ ]:
# Validação final do Notebook 02
assert solo_processado.duplicated(subset=["codigo_ibge", "ano"]).sum() == 0
assert solo_curated["ano"].between(2019, 2024).all()
assert solo_curated["codigo_ibge"].notna().all()
assert solo_curated["uf"].notna().all()
assert solo_curated["regiao"].notna().all()
assert set(solo_curated["regiao"].unique()) == {"Centro-Oeste", "Sul"}
assert set(solo_curated["uf"].unique()) <= {"DF", "GO", "MT", "MS", "PR", "RS", "SC"}
assert solo_curated["carbono_solo_t_ha"].dropna().ge(0).all()

print("✅ Notebook 02 concluído com sucesso.")
print("✅ PROCESSED e CURATED restritos a Centro-Oeste e Sul.")
